# Projet F11 : Volatilité du change euro-dollar

**Durée estimée :** environ 2 semaines · **Niveau :** maîtrise · **Données :** FRED, série DEXUSEU (dollars américains pour un euro)

## Objectifs

À la fin de ce notebook, vous saurez :

1. justifier le **type d'apprentissage** (supervisé, régression sur série temporelle) et définir une cible de volatilité ;
2. nettoyer une série financière quotidienne (jours fériés manquants) et calculer des **rendements logarithmiques** ;
3. construire des variables à partir du passé **sans fuite temporelle** ;
4. comparer une référence simple, une régression linéaire et un gradient boosting avec une **validation chronologique** ;
5. évaluer avec le **RMSE**, le **MAE** et le **QLIKE**, une mesure adaptée à la volatilité ;
6. comparer à un modèle **GARCH(1,1)** (partie avancée) et conclure avec prudence.

## Question de recherche

> Les variations passées du taux de change permettent-elles de prévoir l'amplitude des variations à venir ?

## Type d'apprentissage

C'est un problème **supervisé** : on dispose d'une cible observée (l'amplitude du rendement du jour) et de variables construites à partir du passé. La cible est continue, donc c'est une **régression**. Comme les observations sont ordonnées dans le temps, on utilise un découpage **chronologique** et jamais aléatoire.

## 1. Importation des bibliothèques

In [ ]:
import os, glob, sys, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
GRAINE = 42
np.random.seed(GRAINE)

## 2. Chargement des données

La série **DEXUSEU** donne le taux de change quotidien : combien de dollars valent un euro. Deux options :

- **Option A (par défaut)** : téléverser le fichier `DEXUSEU.csv` fourni avec le projet ;
- **Option B** : télécharger la série directement depuis FRED (historique complet depuis 1999). Mettez `SOURCE = "url"` dans la cellule suivante.

In [ ]:
SOURCE = "fichier"   # "fichier" (téléversement) ou "url" (téléchargement depuis FRED)
URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DEXUSEU&cosd=1999-01-04"

if SOURCE == "url":
    brut = pd.read_csv(URL)
else:
    candidats = glob.glob("DEXUSEU*.csv")
    if not candidats:
        from google.colab import files
        print("Sélectionnez le fichier DEXUSEU.csv")
        televerse = files.upload()
        candidats = list(televerse.keys())
    brut = pd.read_csv(candidats[0])

# Noms de colonnes uniformes, conversion des types ("." ou vide = valeur manquante)
brut.columns = ["date", "taux"]
brut["date"] = pd.to_datetime(brut["date"])
brut["taux"] = pd.to_numeric(brut["taux"], errors="coerce")
brut = brut.set_index("date").sort_index()
print("Dimensions :", brut.shape)
print("Période    :", brut.index.min().date(), "→", brut.index.max().date())
brut.head()

## 3. Première exploration

On vérifie d'abord les valeurs manquantes. Dans FRED, elles correspondent aux **jours fériés américains** où le marché de référence n'a pas publié de taux. Ce ne sont pas des erreurs de saisie : on retire ces jours.

In [ ]:
print("Valeurs manquantes :", brut["taux"].isna().sum(), "sur", len(brut),
      f"({brut['taux'].isna().mean():.1%})")
print()
brut["taux"].describe().round(4)

In [ ]:
taux = brut["taux"].dropna()
print("Observations conservées :", len(taux))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(taux.index, taux, lw=1)
ax.set_title("Taux de change euro-dollar (USD pour 1 EUR)")
ax.set_xlabel("Date"); ax.set_ylabel("USD par EUR")
plt.tight_layout(); plt.show()

**Lecture.** Le niveau du taux de change n'est pas stationnaire : il dérive sur de longues périodes. On ne modélise donc pas le niveau, mais les **rendements**, qui sont des variations relatives.

## 4. Rendements et mesure de la volatilité

### 4.1 Rendements logarithmiques

$$r_t = 100 \times \ln\!\left(\frac{S_t}{S_{t-1}}\right)$$

Le facteur 100 exprime le rendement en pourcentage. La **volatilité** est l'écart-type (non observable) des rendements. On n'en observe qu'un **proxy bruité** : la valeur absolue $|r_t|$. C'est la cible de ce projet.

In [ ]:
rend = 100 * np.log(taux).diff().dropna()      # rendement quotidien en %
rend_abs = rend.abs()                          # proxy de volatilité (cible)

print(f"Rendement moyen quotidien : {rend.mean():.4f} %")
print(f"Écart-type quotidien      : {rend.std():.4f} %")
print(f"Amplitude absolue moyenne : {rend_abs.mean():.4f} %")
print(f"Asymétrie : {rend.skew():.2f} | Kurtosis en excès : {rend.kurt():.2f}  (0 pour une loi normale)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(rend.index, rend, lw=0.6)
axes[0].set_title("Rendements quotidiens (%)")
axes[0].set_xlabel("Date")

sns.histplot(rend, bins=60, stat="density", alpha=0.5, ax=axes[1])
x = np.linspace(rend.min(), rend.max(), 300)
axes[1].plot(x, np.exp(-0.5 * ((x - rend.mean()) / rend.std())**2) / (rend.std() * np.sqrt(2 * np.pi)),
             color="crimson", label="loi normale ajustée")
axes[1].set_title("Distribution des rendements")
axes[1].set_xlabel("Rendement (%)"); axes[1].legend()
plt.tight_layout(); plt.show()

**Lecture.** Les rendements oscillent autour de zéro, mais l'amplitude des oscillations change : des périodes calmes alternent avec des périodes agitées. La distribution a des **queues plus épaisses** que la loi normale (kurtosis en excès positif) : les grands mouvements sont plus fréquents que ne le prévoit une loi normale.

### 4.2 La volatilité est-elle persistante ?

Si la volatilité se regroupe en périodes (*volatility clustering*), alors $|r_t|$ doit être corrélé à $|r_{t-k}|$, alors que $r_t$ lui-même ne l'est presque pas. On le vérifie avec l'autocorrélation.

In [ ]:
retards = range(1, 31)
acf_r   = [rend.autocorr(k) for k in retards]
acf_abs = [rend_abs.autocorr(k) for k in retards]
seuil = 1.96 / np.sqrt(len(rend))     # seuil approximatif de significativité à 5 %

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, acf, titre in zip(axes, [acf_r, acf_abs],
                          ["Autocorrélation des rendements $r_t$", "Autocorrélation des amplitudes $|r_t|$"]):
    ax.bar(list(retards), acf, width=0.6)
    ax.axhline(seuil, color="crimson", ls="--", lw=1); ax.axhline(-seuil, color="crimson", ls="--", lw=1)
    ax.set_title(titre); ax.set_xlabel("Retard (jours)")
axes[0].set_ylabel("Autocorrélation")
plt.tight_layout(); plt.show()

**Lecture.** Les barres dépassant les pointillés sont significativement non nulles. Les rendements sont peu corrélés entre eux (la direction est difficile à prévoir), tandis que les amplitudes sont corrélées sur de nombreux retards : **l'amplitude est partiellement prévisible, la direction ne l'est pas.** C'est ce qui justifie ce projet.

## 5. Construction des variables

Pour prévoir $|r_t|$, on n'utilise que l'information disponible **avant** $t$ :

| Variable | Contenu |
|---|---|
| `abs_lag1` … `abs_lag5` | amplitudes des 5 jours précédents |
| `moy5`, `moy20`, `moy60` | moyennes des amplitudes sur 5, 20 et 60 jours **précédents** |

**Règle anti-fuite.** Chaque moyenne glissante est calculée sur la série décalée d'un jour (`shift(1)`) : le jour $t$ n'entre jamais dans ses propres variables. C'est l'erreur la plus fréquente avec les séries temporelles.

In [ ]:
donnees = pd.DataFrame({"cible": rend_abs})
for k in range(1, 6):
    donnees[f"abs_lag{k}"] = rend_abs.shift(k)
passe = rend_abs.shift(1)                      # information disponible à la veille
donnees["moy5"]  = passe.rolling(5).mean()
donnees["moy20"] = passe.rolling(20).mean()
donnees["moy60"] = passe.rolling(60).mean()
donnees = donnees.dropna()

# Vérification explicite de l'absence de fuite : lag1 au jour t = cible au jour t-1
assert np.allclose(donnees["abs_lag1"].iloc[1:].values, donnees["cible"].iloc[:-1].values)
print("Vérification anti-fuite réussie.")
print("Observations utilisables :", len(donnees))
donnees.head()

## 6. Découpage chronologique

On réserve les **20 % les plus récents** pour le test. Le test simule l'usage réel : prévoir un futur que le modèle n'a jamais vu. Un découpage aléatoire mélangerait passé et futur et gonflerait artificiellement les performances.

In [ ]:
cible = "cible"
variables = [c for c in donnees.columns if c != cible]

n_train = int(len(donnees) * 0.80)
train, test = donnees.iloc[:n_train], donnees.iloc[n_train:]
X_train, y_train = train[variables], train[cible]
X_test,  y_test  = test[variables],  test[cible]

print(f"Entraînement : {len(train)} obs. ({train.index[0].date()} → {train.index[-1].date()})")
print(f"Test         : {len(test)} obs. ({test.index[0].date()} → {test.index[-1].date()})")

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(train.index, train[cible], lw=0.5, label="entraînement")
ax.plot(test.index,  test[cible],  lw=0.5, color="darkorange", label="test")
ax.set_title("Cible $|r_t|$ : découpage chronologique"); ax.legend()
plt.tight_layout(); plt.show()

## 7. Métriques d'évaluation

Soit $y_t = |r_t|$ la cible et $\hat y_t$ sa prévision.

- **RMSE** $=\sqrt{\overline{(y_t-\hat y_t)^2}}$ et **MAE** $=\overline{|y_t-\hat y_t|}$ : erreurs en points de pourcentage, faciles à lire.
- **QLIKE** : on convertit la prévision d'amplitude en prévision de **variance** $h_t$, puis

$$\text{QLIKE}=\overline{\ln h_t+\frac{r_t^2}{h_t}}$$

Pour une loi normale, $E|r| = \sigma\sqrt{2/\pi}$, donc $h_t = \left(\hat y_t\sqrt{\pi/2}\right)^2$. Le QLIKE est la mesure de perte standard pour la volatilité car elle reste **fiable même quand la variance réalisée est mesurée avec du bruit**, ce qui est notre cas : $r_t^2$ n'est qu'un échantillon unique. Elle **pénalise davantage une sous-estimation** de la variance qu'une surestimation de même taille, ce qui correspond à l'asymétrie du risque réel. **Plus c'est bas, mieux c'est.**

In [ ]:
def metriques(y, pred):
    y, pred = np.asarray(y), np.clip(np.asarray(pred), 1e-3, None)   # une volatilité est positive
    h = (pred * np.sqrt(np.pi / 2)) ** 2                              # variance prévue
    return {"RMSE": np.sqrt(np.mean((y - pred) ** 2)),
            "MAE":  np.mean(np.abs(y - pred)),
            "QLIKE": np.mean(np.log(h) + y ** 2 / h)}

class MoyenneGlissante20(BaseEstimator, RegressorMixin):
    # Référence « intelligente » : prévoir la moyenne des 20 derniers jours.
    def fit(self, X, y=None):
        return self
    def predict(self, X):
        return X["moy20"].to_numpy()

## 8. Modèles et validation chronologique

### 8.1 Les modèles

1. **Référence : volatilité historique moyenne.** Prévoit toujours la moyenne de $|r|$ sur l'entraînement. Tout modèle utile doit la battre.
2. **Moyenne glissante 20 jours.** Référence plus exigeante : elle s'adapte déjà au niveau récent de volatilité.
3. **Régression linéaire** sur les variables retardées (standardisées).
4. **Gradient boosting** avec des arbres peu profonds et un faible taux d'apprentissage, pour limiter le surapprentissage.

### 8.2 Validation croisée chronologique

`TimeSeriesSplit` entraîne toujours sur le passé et valide sur la période qui suit. Le paramètre `gap=5` laisse un écart de 5 jours entre les deux, car les variables de validation recouvrent les derniers jours d'entraînement. La validation ne touche **jamais** au jeu de test.

In [ ]:
modeles = {
    "Référence : moyenne historique": DummyRegressor(strategy="mean"),
    "Moyenne glissante 20 j": MoyenneGlissante20(),
    "Régression linéaire": Pipeline([("echelle", StandardScaler()), ("modele", LinearRegression())]),
    "Gradient boosting": GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.03,
                                                   subsample=0.8, random_state=GRAINE),
}

validation = TimeSeriesSplit(n_splits=5, gap=5)
lignes = []
for nom, modele in modeles.items():
    scores = []
    for idx_ent, idx_val in validation.split(X_train):
        modele.fit(X_train.iloc[idx_ent], y_train.iloc[idx_ent])
        scores.append(metriques(y_train.iloc[idx_val], modele.predict(X_train.iloc[idx_val])))
    scores = pd.DataFrame(scores)
    lignes.append({"Modèle": nom,
                   "RMSE": scores["RMSE"].mean(), "MAE": scores["MAE"].mean(),
                   "QLIKE": scores["QLIKE"].mean(), "Écart-type QLIKE": scores["QLIKE"].std()})

tableau_cv = pd.DataFrame(lignes).sort_values("QLIKE").reset_index(drop=True)
tableau_cv.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
couleurs = sns.color_palette("Set2", len(tableau_cv))
for ax, m in zip(axes, ["RMSE", "MAE", "QLIKE"]):
    ax.barh(tableau_cv["Modèle"], tableau_cv[m], color=couleurs)
    ax.set_title(f"{m} (validation chronologique)")
    ax.set_xlim(tableau_cv[m].min() * 0.95, tableau_cv[m].max() * 1.02)
    ax.invert_yaxis()
axes[1].set_yticklabels([]); axes[2].set_yticklabels([])
plt.tight_layout(); plt.show()

**Lecture.** Comparez chaque modèle à la **référence** et à la **moyenne glissante 20 jours**. Sur ces données, la régression linéaire est légèrement meilleure que la référence, alors que le **gradient boosting est moins bon** en validation : avec environ 950 observations très bruitées, un modèle flexible apprend du bruit (surapprentissage). Les écarts-types, élevés, montrent que les plis diffèrent beaucoup entre eux. Un modèle plus simple qui égale ou bat un modèle complexe est un résultat normal, pas un échec.

## 9. Évaluation finale sur le jeu de test

Chaque modèle est entraîné sur **tout** l'entraînement puis évalué **une seule fois** sur le test. On ne retouche pas les modèles après avoir vu ces résultats : ce serait du *data snooping*.

In [ ]:
predictions = {}
lignes = []
for nom, modele in modeles.items():
    modele.fit(X_train, y_train)
    predictions[nom] = np.clip(modele.predict(X_test), 1e-3, None)
    lignes.append({"Modèle": nom, **metriques(y_test, predictions[nom])})

tableau_test = pd.DataFrame(lignes).sort_values("QLIKE").reset_index(drop=True)
ref = tableau_test.loc[tableau_test["Modèle"].str.startswith("Référence"), "RMSE"].iloc[0]
tableau_test["Gain RMSE vs référence"] = (1 - tableau_test["RMSE"] / ref).map("{:.1%}".format)
tableau_test.round(4)

### 9.1 Volatilité prévue contre volatilité réalisée

Le graphique de gauche montre les valeurs brutes : la cible $|r_t|$ est très irrégulière. Pour juger la qualité de la prévision, le graphique de droite **lisse les deux séries sur 20 jours**. Ce lissage ne sert qu'à lire le graphique : les métriques ci-dessus utilisent les valeurs brutes.

In [ ]:
meilleur = tableau_test.iloc[0]["Modèle"]
vol_prevue = pd.Series(predictions[meilleur], index=y_test.index)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(y_test.index, y_test, lw=0.6, color="grey", alpha=0.8, label="réalisée $|r_t|$ (brute)")
axes[0].plot(vol_prevue.index, vol_prevue, lw=1.4, color="crimson", label=f"prévue : {meilleur}")
axes[0].set_title("Valeurs quotidiennes"); axes[0].set_ylabel("Amplitude (%)"); axes[0].legend()

axes[1].plot(y_test.index, y_test.rolling(20).mean(), lw=1.6, color="black", label="réalisée (moyenne 20 j)")
axes[1].plot(vol_prevue.index, vol_prevue.rolling(20).mean(), lw=1.6, color="crimson", label="prévue (moyenne 20 j)")
axes[1].set_title("Séries lissées sur 20 jours"); axes[1].set_ylabel("Amplitude (%)"); axes[1].legend()
plt.tight_layout(); plt.show()

**Lecture.** La prévision varie peu autour d'un niveau moyen et ne capte pas les pics ponctuels : un choc soudain ne peut pas être annoncé par le passé. **Attention à l'interprétation du gain sur la référence** : la volatilité de la période de test est plus faible que celle de l'entraînement. Une partie de l'avantage des modèles qui s'adaptent au niveau récent vient donc de ce simple changement de niveau, pas d'une vraie prévision jour par jour. La section 9.2 permet de vérifier ce point.

### 9.2 Prévision contre réalité, jour par jour

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(vol_prevue, y_test, s=14, alpha=0.5)
lim = max(y_test.max(), vol_prevue.max())
ax.plot([0, lim], [0, lim], color="crimson", ls="--", label="prévision parfaite")
ax.set_xlabel("Amplitude prévue (%)"); ax.set_ylabel("Amplitude réalisée (%)")
ax.set_title(f"Réalisé contre prévu : {meilleur}"); ax.legend()
plt.tight_layout(); plt.show()
print(f"Corrélation prévu / réalisé : {np.corrcoef(vol_prevue, y_test)[0, 1]:.3f}")

**Lecture.** La corrélation entre prévu et réalisé est très faible, ce qui confirme que le pouvoir prédictif jour par jour est limité. Le nuage est très dispersé verticalement : pour une même prévision, l'amplitude réalisée varie beaucoup. C'est le **bruit du proxy** $|r_t|$ évoqué dans la fiche du projet. Il explique pourquoi une erreur absolue reste élevée même avec un bon modèle.

## 10. Les gains sont-ils significatifs ?

Avec environ 240 jours de test, un écart de quelques pourcents entre deux modèles peut venir du hasard. On estime un intervalle de confiance de la **différence d'erreur quadratique** entre le meilleur modèle et la référence par **bootstrap par blocs** : on rééchantillonne des blocs de 10 jours consécutifs pour conserver la dépendance temporelle.

In [ ]:
def bootstrap_blocs(diff, taille_bloc=10, n_rep=2000, graine=GRAINE):
    # IC à 95 % de la moyenne d'une série de différences, par blocs consécutifs
    rng = np.random.default_rng(graine)
    n = len(diff)
    n_blocs = int(np.ceil(n / taille_bloc))
    moyennes = []
    for _ in range(n_rep):
        debuts = rng.integers(0, n - taille_bloc + 1, size=n_blocs)
        echantillon = np.concatenate([diff[d:d + taille_bloc] for d in debuts])[:n]
        moyennes.append(echantillon.mean())
    return np.percentile(moyennes, [2.5, 97.5])

nom_ref = "Référence : moyenne historique"
y_arr = y_test.to_numpy()
err_ref = (y_arr - predictions[nom_ref]) ** 2

for nom in predictions:
    if nom == nom_ref:
        continue
    diff = err_ref - (y_arr - predictions[nom]) ** 2      # > 0 : le modèle fait mieux que la référence
    bas, haut = bootstrap_blocs(diff)
    verdict = "gain significatif" if bas > 0 else ("perte significative" if haut < 0 else "écart non significatif")
    print(f"{nom:<28} gain moyen d'erreur² : {diff.mean():+.4f}  IC95 % [{bas:+.4f} ; {haut:+.4f}]  → {verdict}")

**Lecture.** Si l'intervalle contient 0, on ne peut pas affirmer que le modèle bat la référence sur cette période de test. Un gain significatif sur la référence **constante** peut simplement refléter l'adaptation au niveau de volatilité récent (voir 9.1) : c'est pourquoi la moyenne glissante 20 jours est la comparaison la plus exigeante. Comme en finance on teste souvent plusieurs modèles, un « gain » isolé doit être accueilli avec prudence.

## 11. Quelles variables comptent ?

L'**importance par permutation** mesure de combien l'erreur augmente quand on mélange une variable. Une variable inutile n'a aucun effet. On l'applique ici au gradient boosting, sur le jeu de test.

In [ ]:
gb = modeles["Gradient boosting"]
imp = permutation_importance(gb, X_test, y_test, n_repeats=30, random_state=GRAINE,
                             scoring="neg_root_mean_squared_error")
imp_df = pd.DataFrame({"variable": variables, "importance": imp.importances_mean,
                       "ecart_type": imp.importances_std}).sort_values("importance")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(imp_df["variable"], imp_df["importance"], xerr=imp_df["ecart_type"], capsize=3)
ax.set_xlabel("Hausse du RMSE quand la variable est mélangée")
ax.set_title("Importance par permutation (gradient boosting, jeu de test)")
plt.tight_layout(); plt.show()

**Lecture.** Les importances sont **proches de zéro et leurs écarts-types sont du même ordre de grandeur** : aucune variable ne ressort nettement. Le signal de prévision contenu dans les amplitudes passées est faible par rapport au bruit, ce qui concorde avec la faible corrélation prévu/réalisé de la section 9.2. Ne commentez pas le classement des variables comme s'il était établi : il n'est pas statistiquement distinguable.

## 12. Comparaison avancée : modèle GARCH(1,1)

Le **GARCH(1,1)** est le modèle de référence de la volatilité financière :

$$\sigma_t^2=\omega+\alpha\,\varepsilon_{t-1}^2+\beta\,\sigma_{t-1}^2$$

La variance du jour dépend du dernier choc ($\alpha$) et de la variance de la veille ($\beta$). Les paramètres sont **estimés sur l'entraînement seulement**, puis figés pour calculer la variance prévue sur le test (prévision à un jour, sans réestimation).

Cette partie requiert la bibliothèque `arch` ; elle s'installe automatiquement dans Colab. Si l'installation échoue, le reste du notebook n'est pas affecté.

In [ ]:
try:
    try:
        from arch import arch_model
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arch"], check=True)
        from arch import arch_model

    date_debut_test = X_test.index[0]
    rend_train = rend.loc[rend.index < date_debut_test]

    # 1) estimation sur l'entraînement uniquement (loi normale, cohérente avec la conversion |r| ↔ σ)
    res = arch_model(rend_train, mean="Constant", vol="GARCH", p=1, q=1, dist="normal").fit(disp="off")
    print(res.params.round(4).to_string())

    # 2) paramètres figés appliqués à toute la série : écart-type conditionnel du jour t
    #    calculé uniquement avec l'information jusqu'à t-1
    res_fixe = arch_model(rend, mean="Constant", vol="GARCH", p=1, q=1, dist="normal").fix(res.params)
    sigma = res_fixe.conditional_volatility.reindex(X_test.index)
    pred_garch = np.clip(sigma.to_numpy() * np.sqrt(2 / np.pi), 1e-3, None)   # E|r| = σ√(2/π)

    predictions["GARCH(1,1)"] = pred_garch
    tableau_final = pd.concat([tableau_test.iloc[:, :4],
                               pd.DataFrame([{"Modèle": "GARCH(1,1)", **metriques(y_test, pred_garch)}])],
                              ignore_index=True).sort_values("QLIKE").reset_index(drop=True)
    display(tableau_final.round(4))
except Exception as e:
    print("Partie GARCH ignorée :", e)
    tableau_final = tableau_test.iloc[:, :4]

**Lecture.** Persistance $=\alpha+\beta$ : proche de 1, elle indique que les chocs de volatilité s'estompent lentement. Le GARCH est conçu pour la volatilité et est souvent très compétitif en QLIKE ; il impose cependant une forme paramétrique précise, alors que le gradient boosting n'en suppose aucune. Cette comparaison est celle qu'attend un rapport de maîtrise.

## 13. Limites et conclusion

**Limites à mentionner dans le rapport.**

1. **Proxy bruité.** $|r_t|$ n'est pas la volatilité : c'est un tirage unique dans une distribution. Une erreur faible n'est pas une bonne prévision de la **direction** du taux, que ce projet ne prévoit pas.
2. **Période de test courte.** Environ 240 jours et un seul régime de marché : les conclusions ne se généralisent pas sans réserve. Avec `SOURCE = "url"`, on obtient l'historique complet depuis 1999 et plusieurs régimes (crise de 2008, dette européenne, 2020).
3. **Données journalières de FRED.** Taux de midi à New York : les mouvements asiatiques et européens de la veille y sont déjà intégrés.
4. **Jours fériés retirés.** Un rendement après un jour férié couvre deux jours calendaires : il est légèrement plus ample en moyenne.
5. **Multiplicité des essais.** Plus on compare de modèles, plus le risque de retenir un gain dû au hasard augmente. D'où le test de la section 10.

**Conclusion à rédiger.** Complétez avec vos résultats :

- La volatilité du change est-elle prévisible à partir de son passé ? (appuyez-vous sur la section 4.2 et sur le tableau de test)
- Quel modèle l'emporte selon le RMSE ? selon le QLIKE ? Les classements diffèrent-ils, et pourquoi ?
- Le gain sur la référence est-il statistiquement significatif ?

## 14. Pistes d'approfondissement

- Passer à l'historique complet depuis 1999 et refaire l'analyse par sous-période.
- Remplacer la cible $|r_t|$ par $r_t^2$ ou par la volatilité réalisée sur 5 jours, moins bruitée.
- Tester un GARCH asymétrique (GJR-GARCH) et la loi de Student.
- Utiliser une validation *walk-forward* avec réestimation périodique.